In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_4.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common infrastructure for all Exercise 4 technique files:
#   - AG News loading + caching (120K train, 7.6K test)
#   - Vocabulary building and text-to-index conversion
#   - ExperimentTracker + ModelRegistry + OnnxBridge setup
#   - Shared constants, device detection, visualisation helpers
#
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import math
from collections import Counter
from pathlib import Path
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml import OnnxBridge

# ════════════════════════════════════════════════════════════════════════
# Environment + Device
# ════════════════════════════════════════════════════════════════════════
setup_environment()

torch.manual_seed(42)
np.random.seed(42)
DEVICE = get_device()

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
MAX_LEN = 40
VAL_FRACTION = 0.05  # share of the AG News TRAIN split held out for model selection
VAL_SPLIT_SEED = 42
VOCAB_SIZE = 15000
EPOCHS_SCRATCH = 8
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN = 64
BERT_EPOCHS = 3
BERT_LR = 2e-5
BERT_BATCH_SIZE = 32


# ════════════════════════════════════════════════════════════════════════
# Core Attention Function (shared across 01, 02, 05)
# ════════════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute scaled dot-product attention.

    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V

    Args:
        q: Query tensor of shape (B, L_q, d_k)
        k: Key tensor of shape (B, L_k, d_k)
        v: Value tensor of shape (B, L_k, d_v)
        mask: Optional mask of shape (B, L_q, L_k). Positions with 0 are
              masked out (set to -inf before softmax).

    Returns:
        (output, attention_weights) where output has shape (B, L_q, d_v)
        and attention_weights has shape (B, L_q, L_k).
    """
    d_k = q.size(-1)
    scores = torch.einsum("bqd,bkd->bqk", q, k) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = torch.einsum("bqk,bkd->bqd", weights, v)
    return out, weights


# ════════════════════════════════════════════════════════════════════════
# AG News Loading
# ════════════════════════════════════════════════════════════════════════
def load_ag_news_split(split: str, cache_name: str) -> pl.DataFrame:
    """Load AG News split, caching to parquet for subsequent runs."""
    cache = Path.cwd() / "data" / "mlfp05" / cache_name
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        return pl.read_parquet(cache)
    print(f"  Downloading AG News {split} from HuggingFace...")
    ds = load_dataset("fancyzhx/ag_news", split=split)
    df = pl.from_pandas(ds.to_pandas())
    df.write_parquet(cache)
    return df


def load_ag_news() -> tuple[pl.DataFrame, pl.DataFrame]:
    """Load full AG News train + test splits with status reporting."""
    print("\n== Loading FULL AG News (120K train + 7.6K test) ==")
    train_df = load_ag_news_split("train", "ag_news_full_train.parquet")
    test_df = load_ag_news_split("test", "ag_news_full_test.parquet")
    print(f"  train rows: {len(train_df):,}   test rows: {len(test_df):,}")
    print(f"  sample headline: {train_df['text'][0][:80]!r}")
    print(f"  class balance (train): {dict(Counter(train_df['label'].to_list()))}")
    return train_df, test_df


# ════════════════════════════════════════════════════════════════════════
# Vocabulary + Tokenisation (for from-scratch models: LSTM + Transformer)
# ════════════════════════════════════════════════════════════════════════
def build_vocab(texts: list[str], max_vocab: int = VOCAB_SIZE) -> dict[str, int]:
    """Build a word-level vocabulary from text list."""
    words: Counter[str] = Counter()
    for t in texts:
        words.update(t.lower().split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in words.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab


def text_to_indices(
    text: str, vocab: dict[str, int], max_len: int = MAX_LEN
) -> list[int]:
    """Convert text to padded index sequence."""
    tokens = text.lower().split()[:max_len]
    idxs = [vocab.get(t, 1) for t in tokens]
    return (idxs + [0] * (max_len - len(idxs)))[:max_len]


def prepare_dataloaders(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    vocab: dict[str, int],
    batch_size: int = 128,
) -> tuple[
    DataLoader, DataLoader, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor
]:
    """Tokenise AG News and build DataLoaders for from-scratch models.

    A fixed 5% of the TRAINING split is held out as the validation set:
    train_model() picks its best epoch on it, so the test split is never
    used for model selection and stays an unbiased final measurement
    (see evaluate_accuracy).

    Returns: (train_loader, val_loader, train_t, train_y, test_t, test_y)
    where train_t/train_y exclude the validation rows.
    """
    train_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in train_df["text"].to_list()],
        dtype=np.int64,
    )
    train_labels = np.array(train_df["label"].to_list(), dtype=np.int64)
    test_tokens = np.array(
        [text_to_indices(t, vocab, MAX_LEN) for t in test_df["text"].to_list()],
        dtype=np.int64,
    )
    test_labels = np.array(test_df["label"].to_list(), dtype=np.int64)

    perm = np.random.default_rng(VAL_SPLIT_SEED).permutation(len(train_tokens))
    n_val = int(round(VAL_FRACTION * len(train_tokens)))
    val_idx, fit_idx = perm[:n_val], perm[n_val:]

    train_t = torch.from_numpy(train_tokens[fit_idx]).to(DEVICE)
    train_y = torch.from_numpy(train_labels[fit_idx]).to(DEVICE)
    val_t = torch.from_numpy(train_tokens[val_idx]).to(DEVICE)
    val_y = torch.from_numpy(train_labels[val_idx]).to(DEVICE)
    test_t = torch.from_numpy(test_tokens).to(DEVICE)
    test_y = torch.from_numpy(test_labels).to(DEVICE)

    train_loader = DataLoader(
        TensorDataset(train_t, train_y), batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(val_t, val_y), batch_size=batch_size)

    return train_loader, val_loader, train_t, train_y, test_t, test_y


# ════════════════════════════════════════════════════════════════════════
# Kailash-ML Engine Setup
# ════════════════════════════════════════════════════════════════════════
async def _setup_engines_async() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]
):
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) + ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transformers.db"
    registry_db = "sqlite:///mlfp05_transformers_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transformers", registry, True


def setup_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
    OnnxBridge,
]:
    """Set up all kailash-ml engines (sync wrapper).

    Returns: (conn, tracker, exp_name, registry, has_registry, bridge)
    """
    conn, tracker, exp_name, registry, has_registry = asyncio.run(
        _setup_engines_async()
    )
    bridge = OnnxBridge()
    return conn, tracker, exp_name, registry, has_registry, bridge


# ════════════════════════════════════════════════════════════════════════
# Training Utilities
# ════════════════════════════════════════════════════════════════════════
async def train_model_async(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Train a PyTorch classifier and log every epoch to ExperimentTracker.

    Uses the modern ``tracker.track(...)`` async context manager -- bulk
    param logging, per-step metrics, automatic COMPLETED/FAILED state.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_losses: list[float] = []
    val_accs: list[float] = []
    best_acc = 0.0
    best_state: dict | None = None
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    async with tracker.track(experiment=exp_name, run_name=model_name) as run:
        await run.log_params(
            {
                "model_type": model_name,
                "epochs": str(epochs),
                "lr": str(lr),
                "dataset_size": str(len(train_loader.dataset)),
                "batch_size": str(train_loader.batch_size),
                "trainable_params": str(param_count),
            }
        )

        for epoch in range(epochs):
            model.train()
            batch_losses = []
            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                batch_losses.append(loss.item())
            scheduler.step()
            epoch_loss = float(np.mean(batch_losses))
            train_losses.append(epoch_loss)

            model.eval()
            with torch.no_grad():
                correct = 0
                total = 0
                for xb, yb in val_loader:
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
                acc = correct / total
                val_accs.append(acc)

            await run.log_metrics(
                {"train_loss": epoch_loss, "val_accuracy": acc},
                step=epoch + 1,
            )
            if acc > best_acc:
                best_acc = acc
                best_state = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
            print(
                f"  [{model_name}] epoch {epoch+1}/{epochs}  "
                f"loss={epoch_loss:.4f}  val_acc={acc:.3f}"
            )

        await run.log_metrics(
            {
                "best_val_accuracy": best_acc,
                "final_train_loss": train_losses[-1],
            }
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_losses, val_accs


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = EPOCHS_SCRATCH,
    lr: float = 2e-3,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model, model_name, train_loader, val_loader, tracker, exp_name, epochs, lr
        )
    )


def evaluate_accuracy(
    model: nn.Module, tokens: torch.Tensor, labels: torch.Tensor, batch_size: int = 512
) -> float:
    """Accuracy of `model` on (tokens, labels) — use it on the TEST split once,
    after train_model() has restored the validation-selected checkpoint."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for start in range(0, len(tokens), batch_size):
            preds = model(tokens[start : start + batch_size]).argmax(dim=-1)
            correct += int((preds == labels[start : start + batch_size]).sum().item())
    return correct / len(tokens)


# ════════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ════════════════════════════════════════════════════════════════════════
def create_attention_heatmap(
    attn_weights: np.ndarray,
    labels: list[str],
    title: str = "Self-Attention Heatmap",
    max_tokens: int = 15,
) -> go.Figure:
    """Create a plotly heatmap from an attention weight matrix.

    Args:
        attn_weights: 2D numpy array of shape (seq, seq)
        labels: token labels for axes
        title: chart title
        max_tokens: limit display to first N non-pad tokens

    Returns:
        plotly Figure
    """
    show_len = min(max_tokens, len([w for w in labels if w != "<pad>"]))
    fig = go.Figure(
        data=go.Heatmap(
            z=attn_weights[:show_len, :show_len],
            x=labels[:show_len],
            y=labels[:show_len],
            colorscale="Viridis",
            colorbar={"title": "Attention weight"},
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Key position",
        yaxis_title="Query position",
        width=600,
        height=500,
    )
    return fig


def get_viz() -> ModelVisualizer:
    """Return a ModelVisualizer instance.

    ModelVisualizer carries a kailash-ml P2 experimental notice (a
    UserWarning at construction); exercises run under
    warnings-as-errors, so the notice is acknowledged narrowly at
    this construction site.
    """
    import warnings as _warnings

    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 4.5: Three-Way Comparison + ONNX Export
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Compare LSTM vs Transformer vs BERT side by side on the same dataset
#   - Explain the accuracy hierarchy (BERT >> Transformer > LSTM) and why
#   - Register all models in the ModelRegistry with versioned metrics
#   - Export the best model to ONNX for portable deployment
#   - Visualise training curves for all three architectures
#   - Interpret model predictions with attention heatmaps
#
# PREREQUISITES: All previous ex_4 files (01-04).
# ESTIMATED TIME: ~25 min
# DATASET: AG News — 120,000 real news headlines, 4 classes.
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math
import pickle
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


print(f"Using device: {DEVICE}")



## THEORY — The Architecture Hierarchy


In [ ]:
# This exercise is the payoff of the entire Exercise 4 sequence. We've
# built three architectures with fundamentally different approaches:
#
#   1. LSTM (sequential, no pre-training):
#      Processes tokens one at a time through a hidden state. No
#      pre-trained knowledge -- learns everything from our 120K headlines.
#      The sequential bottleneck limits long-range dependency capture.
#
#   2. Transformer (parallel attention, no pre-training):
#      Processes all tokens simultaneously via self-attention. Same
#      training data as LSTM, but the attention mechanism provides
#      direct access to all positions. Still learns from scratch.
#
#   3. BERT (parallel attention + pre-training):
#      Same architecture as the Transformer, but starts with pre-trained
#      weights from billions of words. Fine-tuning adapts this vast
#      language knowledge to our specific task.
#
# The comparison isolates two variables:
#   LSTM -> Transformer: the value of ATTENTION (parallel vs sequential)
#   Transformer -> BERT: the value of PRE-TRAINING (scratch vs transfer)
#
# Together, these reveal why modern NLP is dominated by pre-trained
# transformers: attention + pre-training is the winning combination.



## TASK 1 — Train all three models (reusing architectures from 02-04)


Bidirectional LSTM for text classification (same as 03_lstm_baseline).


Multi-head attention (same as 02_transformer_encoder).


Sinusoidal positional encoding (same as 02_transformer_encoder).


Transformer encoder classifier (same as 02_transformer_encoder).


In [ ]:
train_df, test_df = load_ag_news()
vocab = build_vocab(train_df["text"].to_list())
train_loader, val_loader, train_t, train_y, test_t, test_y = prepare_dataloaders(
    train_df, test_df, vocab
)
conn, tracker, exp_name, registry, has_registry, bridge = setup_engines()

# --- Model Architectures (defined here for standalone execution) ---


# TODO: Define LSTMClassifier — bidirectional LSTM for text classification
# Hint: Same architecture as 03_lstm_baseline.py
# - __init__: embed, lstm (bidirectional), head_drop, head (hidden_dim * 2 -> n_classes)
# - forward: embed -> lstm -> mean pool over non-pad -> dropout -> head
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        hidden_dim: int = 128,
        n_layers: int = 2,
        n_classes: int = 4,
        dropout: float = 0.3,
    ):
        super().__init__()
        ...  # YOUR CODE HERE — nn.Embedding, nn.LSTM(bidirectional=True), nn.Dropout, nn.Linear

    def forward(
        self, tokens: torch.Tensor
    ) -> torch.Tensor: ...  # YOUR CODE HERE — embed -> lstm -> mean pool -> head


# TODO: Define EducationalMultiHead — multi-head attention wrapper
# Hint: Same architecture as 02_transformer_encoder.py
class EducationalMultiHead(nn.Module):

    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        b, seq, d = x.shape
        # TODO: Compute QKV, split heads, apply attention, concatenate, project
        # Hint: (b, seq, 3*d_model) -> q, k, v of (b, seq, n_heads, d_k)
        #   -> fold heads into batch (b*n_heads, seq, d_k) -> attention
        #   -> unfold to (b, seq, d_model) -> self.proj. Return (output,
        #   weights reshaped to (b, n_heads, seq, seq)).
        ...  # YOUR CODE HERE


# TODO: Define PositionalEncoding — sinusoidal positional encoding
# Hint: Same as 02_transformer_encoder.py
class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        # TODO: sin(position * div) into the even feature columns,
        #   cos(position * div) into the odd ones.
        ...  # YOUR CODE HERE
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


# TODO: Define TransformerClassifier — full Transformer encoder classifier
# Hint: Same as 02_transformer_encoder.py
class TransformerClassifier(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 3,
        n_classes: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()
        # TODO: Build architecture — embed, posenc, emb_drop, encoder, head_drop, head
        ...  # YOUR CODE HERE

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        # TODO: embed -> posenc -> dropout -> encoder(with pad mask) -> mean pool -> head
        ...  # YOUR CODE HERE


# --- LSTM ---
print("\n== Training LSTM baseline ==")
# TODO: Create and train LSTM model
# Hint: same configuration and train_model call as 03_lstm_baseline.py
#   (embed/hidden 128, 2 layers, 4 classes, run name "lstm_baseline").
lstm_model = ...  # YOUR CODE HERE
lstm_losses, lstm_accs = ...  # YOUR CODE HERE

# --- Transformer ---
print("\n== Training Transformer ==")
# TODO: Create and train Transformer model
# Hint: same configuration and train_model call as 02_transformer_encoder.py
#   (d_model 128, 4 heads, 3 layers, 4 classes, run name "transformer").
transformer_model = ...  # YOUR CODE HERE
transformer_losses, transformer_accs = ...  # YOUR CODE HERE

# --- BERT ---
print(f"\n== Fine-tuning {BERT_MODEL_NAME} ==")
from transformers import BertTokenizer, BertForSequenceClassification

bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=4
).to(DEVICE)

# TODO: Freeze lower 8 of 12 layers
# Hint: as in 04_bert_finetuning.py — read the layer index out of each
#   "bert.encoder.layer.<k>..." parameter name; freeze k < 8 and the embeddings.
for name, param in bert_model.named_parameters():
    ...  # YOUR CODE HERE

trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in bert_model.parameters())


def tokenise_for_bert(
    texts: list[str], max_len: int = BERT_MAX_LEN
) -> tuple[torch.Tensor, torch.Tensor]:
    encoding = bert_tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    return encoding["input_ids"], encoding["attention_mask"]


print("  Tokenising for BERT...")
bert_train_ids, bert_train_mask = tokenise_for_bert(train_df["text"].to_list())
bert_test_ids, bert_test_mask = tokenise_for_bert(test_df["text"].to_list())
bert_train_y = torch.tensor(train_df["label"].to_list(), dtype=torch.long)
bert_test_y = torch.tensor(test_df["label"].to_list(), dtype=torch.long)

bert_train_loader = DataLoader(
    TensorDataset(
        bert_train_ids.to(DEVICE), bert_train_mask.to(DEVICE), bert_train_y.to(DEVICE)
    ),
    batch_size=BERT_BATCH_SIZE,
    shuffle=True,
)
bert_test_loader = DataLoader(
    TensorDataset(
        bert_test_ids.to(DEVICE), bert_test_mask.to(DEVICE), bert_test_y.to(DEVICE)
    ),
    batch_size=BERT_BATCH_SIZE,
)


async def train_bert_async(model, train_loader, test_loader, epochs=3, lr=2e-5):
    # TODO: Implement BERT training loop with ExperimentTracker
    # Hint: AdamW over trainable parameters only (lr, weight decay 0.01);
    #   the LinearLR scheduler and tracker run below are provided.
    optimizer = ...  # YOUR CODE HERE
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.1,
        total_iters=epochs,
    )
    train_losses, test_accs = [], []

    async with tracker.track(experiment=exp_name, run_name="bert_finetune") as run:
        await run.log_params(
            {
                "model_type": "bert_finetune",
                "base_model": BERT_MODEL_NAME,
                "epochs": str(epochs),
                "lr": str(lr),
                "frozen_layers": "0-7",
                "trainable_params": str(trainable),
                "dataset_size": str(len(train_loader.dataset)),
            }
        )
        for epoch in range(epochs):
            model.train()
            batch_losses = []
            for batch_idx, (ids, mask, labels) in enumerate(train_loader):
                # TODO: Forward + backward + step
                # Hint: labels in -> outputs.loss out; clip grad norm to 1.0;
                #   record each batch loss in batch_losses.
                ...  # YOUR CODE HERE
                if (batch_idx + 1) % 500 == 0:
                    print(
                        f"    batch {batch_idx+1}/{len(train_loader)}  loss={np.mean(batch_losses[-500:]):.4f}"
                    )
            scheduler.step()
            epoch_loss = float(np.mean(batch_losses))
            train_losses.append(epoch_loss)

            model.eval()
            with torch.no_grad():
                correct = total_count = 0
                for ids, mask, labels in test_loader:
                    # TODO: Get predictions and accumulate accuracy
                    #   (argmax of .logits; update correct and total_count).
                    ...  # YOUR CODE HERE
                acc = correct / total_count
                test_accs.append(acc)

            await run.log_metrics(
                {"train_loss": epoch_loss, "test_accuracy": acc}, step=epoch + 1
            )
            print(
                f"  [BERT] epoch {epoch+1}/{epochs}  loss={epoch_loss:.4f}  test_acc={acc:.3f}"
            )

        await run.log_metrics(
            {"final_test_accuracy": test_accs[-1], "final_train_loss": train_losses[-1]}
        )
    return train_losses, test_accs


bert_losses, bert_accs = asyncio.run(
    train_bert_async(bert_model, bert_train_loader, bert_test_loader, epochs=3)
)



## DIAGNOSTIC CHECKPOINT — comparative Prescription Pad for all 3


Cross-entropy on one (token_ids, labels) batch.


In [ ]:
# Same instruments, three architectures, same data. Probes come from the
# training loaders (the probe runs in train mode).
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _ce_loss(m, batch):
    xb, yb = batch
    return F.cross_entropy(m(xb), yb)


def _bert_loss(m, ids, mask, labels):
    return m(input_ids=ids, attention_mask=mask, labels=labels).loss


def _bert_adapter(batch):
    # BERT batches are (ids, mask, labels), passed to the loss as three args
    return batch[0], batch[1], batch[2]


for _title, _model, _loader, _loss, _hist, _adapter, _n in [
    ("LSTM", lstm_model, train_loader, _ce_loss, lstm_losses, None, 8),
    ("Transformer", transformer_model, train_loader, _ce_loss, transformer_losses, None, 8),
    ("BERT fine-tune", bert_model, bert_train_loader, _bert_loss, bert_losses, _bert_adapter, 4),
]:
    print(f"\n── Diagnostic Report ({_title}) ──")
    _diag, _findings = run_diagnostic_checkpoint(
        _model,
        _loader,
        _loss,
        title=f"{_title} (3-way comparison)",
        n_batches=_n,
        train_losses=_hist,
        batch_adapter=_adapter,
        show=False,
    )
    print_prescription_pad(_findings, f"{_title} (3-way comparison)")

# ══════ READING THE THREE PADS (key: see ex_1/01_standard_ae.py) ══════
# Line the three pads up. They read optimisation health per layer and
# the training-loss trend; the accuracy, size and speed table below is
# what ranks the models. BERT's frozen layers 0-7 receive no gradient
# by design. Readings you can tie to the architecture (the LSTM's
# recurrent weights vs the Transformer's residual stack) are worth
# writing down; differences you cannot explain are worth re-running.



In [ ]:
# From-scratch models: the validation-selected checkpoint, measured once
# on the test split. BERT: its per-epoch test numbers were only monitored
# (no selection), so report the final epoch.
lstm_test_acc = evaluate_accuracy(lstm_model, test_t, test_y)
transformer_test_acc = evaluate_accuracy(transformer_model, test_t, test_y)
bert_test_acc = bert_accs[-1]



### Checkpoint 1


In [ ]:
assert lstm_test_acc > 0.60, f"LSTM should exceed 60%, got {lstm_test_acc:.3f}"
assert (
    transformer_test_acc > 0.60
), f"Transformer should exceed 60%, got {transformer_test_acc:.3f}"
assert bert_test_acc > 0.85, f"BERT should exceed 85%, got {bert_test_acc:.3f}"
print("\n--- Checkpoint 1 passed --- all three models trained\n")



## TASK 2 — Visualise: Side-by-side comparison table + training curves


In [ ]:
# TODO: Build results dictionary and print comparison table
# Hint: keys "LSTM", "Transformer", "BERT (fine-tuned)"; each value holds
#   "test_acc" (the *_test_acc above), "final_loss" (last training loss) and
#   "params" (total parameter count; BERT's is total_params).
results = ...  # YOUR CODE HERE

print("\n== 3-Way Model Comparison on AG News ==")
print(f"{'Model':<20} {'Best Acc':>10} {'Final Loss':>12} {'Params':>12}")
print("-" * 56)
for name, r in results.items():
    print(
        f"{name:<20} {r['test_acc']:>10.3f} {r['final_loss']:>12.4f} {r['params']:>12,}"
    )

# TODO: Create training curves comparison chart
# Hint: viz.training_history with all 6 series (loss and val accuracy for
#   each model) against "Epoch"; save as HTML to ex_4_5_training_curves.html.
viz = get_viz()
fig_curves = ...  # YOUR CODE HERE
...  # YOUR CODE HERE
print("\nTraining curves saved to ex_4_5_training_curves.html")

# TODO: Sample predictions from all three models
# Hint: tokenise sample texts, run through each model, compare predictions
sample_texts = test_df["text"].to_list()[:5]
sample_true = test_df["label"].to_list()[:5]
sample_idx = torch.tensor(
    [text_to_indices(t, vocab, MAX_LEN) for t in sample_texts],
    dtype=torch.long,
    device=DEVICE,
)

transformer_model.eval()
lstm_model.eval()
bert_model.eval()
with torch.no_grad():
    transformer_preds = (
        ...
    )  # YOUR CODE HERE — predicted class ids as a Python list
    lstm_preds = (
        ...
    )  # YOUR CODE HERE
    bert_sample_ids, bert_sample_mask = tokenise_for_bert(sample_texts)
    bert_preds = (
        ...
    )  # YOUR CODE HERE — BERT takes the bert_sample_* tensors (on DEVICE)

print(f"\n== Sample Predictions (all 3 models) ==")
print(f"{'Headline':<50} {'True':<10} {'LSTM':<10} {'Trans':<10} {'BERT':<10}")
print("-" * 90)
for i, text in enumerate(sample_texts):
    t = CLASS_NAMES[sample_true[i]]
    l = CLASS_NAMES[lstm_preds[i]]
    tr = CLASS_NAMES[transformer_preds[i]]
    b = CLASS_NAMES[bert_preds[i]]
    print(f"{text[:48]:<50} {t:<10} {l:<10} {tr:<10} {b:<10}")



### Checkpoint 2


In [ ]:
best_model_name = max(results, key=lambda k: results[k]["test_acc"])
assert best_model_name == "BERT (fine-tuned)", (
    f"Expected BERT to be the best model, but {best_model_name} won. "
    "Pre-trained models should dominate on standard NLP benchmarks."
)
assert Path("ex_4_5_training_curves.html").exists(), "Training curves should be saved"
# INTERPRETATION: The 3-way comparison reveals a clear hierarchy:
#   BERT >> Transformer > LSTM
# BERT dominates because it starts with pre-trained language knowledge.
# The Transformer edges out the LSTM because attention captures long-range
# dependencies without the information bottleneck of a fixed-size hidden
# state. The LSTM still does respectably -- it is a strong baseline.
print("\n--- Checkpoint 2 passed --- 3-way comparison complete\n")



## TASK 3 — Register all models in ModelRegistry


Register all three models in the ModelRegistry with metrics.


In [ ]:
async def register_all_models():
    if not has_registry:
        print("  ModelRegistry not available -- skipping registration")
        return {}

    from kailash_ml.types import MetricSpec

    model_versions = {}
    models_to_register = [
        ("m5_bert_agnews", bert_model.state_dict(), bert_test_acc, "bert_finetune"),
        (
            "m5_transformer_agnews",
            transformer_model.state_dict(),
            transformer_test_acc,
            "transformer",
        ),
        ("m5_lstm_agnews", lstm_model.state_dict(), lstm_test_acc, "lstm_baseline"),
    ]

    # TODO: Register each model in the registry
    # Hint: the artifact is the pickled state_dict (bytes); register_model is
    #   async and takes a list of MetricSpec (at least test_accuracy). Store
    #   each returned version in model_versions under its model_type.
    for name, state_dict, test_acc, model_type in models_to_register:
        ...  # YOUR CODE HERE

    return model_versions


model_versions  = await register_all_models()



### Checkpoint 3


In [ ]:
if has_registry:
    assert len(model_versions) == 3, "Should register all 3 models"
print("\n--- Checkpoint 3 passed --- models registered in ModelRegistry\n")



## TASK 4 — Export best model (BERT) to ONNX via OnnxBridge


In [ ]:
# In production, the ModelRegistry stores the winning model. OnnxBridge
# exports it to ONNX format for portable deployment (any language, any
# runtime, no PyTorch dependency). This is how production ML pipelines
# separate training (Python) from serving (any language).
onnx_path = Path("ex_4_bert_agnews.onnx")
bert_model.eval()


class BertLogits(nn.Module):

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        mask = (input_ids != 0).long()
        return self.model(input_ids=input_ids, attention_mask=mask).logits


# Export on the CPU, then move BERT back. The sample is a batch of TWO
# headlines, not one: the exporter traces with torch.export, which treats
# size-1 dimensions as constants and can freeze the batch size at 1.
bert_model.cpu()
export_model = BertLogits(bert_model).eval()
# TODO: Export export_model with OnnxBridge ("torch" framework) to
#   onnx_path, tracing with the first two test headlines' token ids.
export_result = ____

# Check the graph on real headlines. OnnxBridge.validate feeds float32
# arrays, so it cannot drive an int64 token-id graph — compare directly.
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path))
check_ids = bert_test_ids[:16]
# TODO: logits for check_ids from the ONNX session (its input name is
#   session.get_inputs()[0].name; feed a numpy array) and from export_model.
onnx_logits = ____
with torch.no_grad():
    torch_logits = ____
bert_model.to(DEVICE)
max_diff = float(np.abs(onnx_logits - torch_logits).max())
agreement = float((onnx_logits.argmax(axis=1) == torch_logits.argmax(axis=1)).mean())
print(
    f"  OnnxBridge.export: success={export_result.success} -> {onnx_path} "
    f"({onnx_path.stat().st_size // 1024:,} KB)"
)
print(f"  ONNX vs PyTorch on 16 test headlines: max |logit diff| = {max_diff:.2e}, "
      f"class agreement = {agreement:.0%}")



### Checkpoint 4


In [ ]:
assert export_result.success, f"OnnxBridge export failed: {export_result.error_message}"
assert agreement == 1.0 and max_diff < 1e-3, "ONNX graph drifted from PyTorch"
# INTERPRETATION: The ModelRegistry gives you a versioned record of every
# model experiment. The ONNX export makes the model portable -- it can run
# on a server without PyTorch installed, in a mobile app, or in a browser
# via ONNX.js. This is how production ML pipelines separate training
# (Python) from serving (any language).
print("\n--- Checkpoint 4 passed --- ONNX export complete\n")



## TASK 5 — Visualise: Attention heatmap from trained Transformer


nn.TransformerEncoderLayer (post-norm, the default) feeds its input
    straight into self_attn, so we rebuild that input (embedding +
    positional encoding) and ask the layer's own attention module for its
    weights. Returns (batch, n_heads, seq, seq); padded keys get weight 0.


In [ ]:
# The attention heatmap is the Transformer's "explanation" -- it shows
# which words the model attends to when classifying a headline.
transformer_model.eval()


def encoder_attention(model: nn.Module, tokens: torch.Tensor) -> torch.Tensor:
    model.eval()
    pad_mask = tokens == 0
    x = model.posenc(model.embed(tokens))
    first_layer = model.encoder.layers[0]
    # TODO: same call as in 02_transformer_encoder.py — the trained layer's
    #   self_attn on x, padded keys masked, per-head weights returned.
    _, weights = ____
    return weights


# Head 0 of the TRAINED first encoder layer (a fresh attention module
# would only show random projections).
# TODO: attention weights for sample_idx[:1], then head 0 as a numpy array
with torch.no_grad():
    attn_weights = ____
    attn_np = ____

words = sample_texts[0].lower().split()[:MAX_LEN]
word_labels = words + ["<pad>"] * (MAX_LEN - len(words))

fig_attn = create_attention_heatmap(
    attn_np,
    word_labels,
    title=f"Transformer Attention on: '{sample_texts[0][:50]}...'",
    max_tokens=15,
)
fig_attn.write_html("ex_4_5_attention_heatmap.html")
print("Attention heatmap saved to ex_4_5_attention_heatmap.html")



### Checkpoint 5


In [ ]:
assert attn_np.shape[0] == MAX_LEN, "Attention heatmap should cover full sequence"
assert Path("ex_4_5_attention_heatmap.html").exists(), "Heatmap should be saved"
print("\n--- Checkpoint 5 passed --- visualisations complete\n")



## DESTINATION-FIRST CLOSE — one-call diagnostics


In [ ]:
# This lesson built attention from scratch, a Transformer encoder, an
# LSTM baseline and a fine-tuned BERT. kailash-ml's
# run_diagnostic_checkpoint is the one call behind each pad: it hooks
# every layer, runs a few probe batches (no weight updates), replays the
# loss history and returns findings plus a DLDiagnostics session whose
# plot_training_dashboard() draws the loss, gradient and activation
# panels. (km.diagnose(model, kind="dl") on its own only builds an
# un-instrumented session, so it has nothing to report.) It does not
# replace the accuracy, latency and deployment comparisons above.
diag, findings = run_diagnostic_checkpoint(
    transformer_model,
    train_loader,
    _ce_loss,
    title="Transformer (close)",
    train_losses=transformer_losses,
    show=False,
)
print_prescription_pad(findings, "Transformer (close)")
dashboard = diag.plot_training_dashboard()  # Plotly figure: dashboard.show()



## REFLECTION


[x] Derived scaled dot-product attention with torch.einsum
  [x] Explained the 1/sqrt(d_k) factor (prevents softmax saturation)
  [x] Wrote a hand-rolled multi-head attention wrapping the scratch kernel
  [x] Built a TransformerClassifier with nn.TransformerEncoder
  [x] Built an LSTM baseline for fair comparison
  [x] Trained all 3 models on FULL AG News (120K headlines)
  [x] Fine-tuned BERT ({BERT_MODEL_NAME}) -- test acc: {bert_test_acc:.1%}
  [x] Visualised attention heatmaps (what the model "looks at")
  [x] Tracked every run with ExperimentTracker (params, per-epoch metrics)
  [x] Registered models in ModelRegistry with versioned metrics
  [x] Exported the fine-tuned model to ONNX for portable deployment

  KEY INSIGHT — The Attention Hierarchy:
    LSTM test acc:        {lstm_test_acc:.1%}  (sequential, no pre-training)
    Transformer test acc: {transformer_test_acc:.1%}  (parallel attention, no pre-training)
    BERT test acc:        {bert_test_acc:.1%}  (parallel attention + pre-training)

  Pre-training is the single biggest lever in NLP. The Transformer
  architecture enables it, but the pre-trained weights are what make
  BERT dominate. This is why modern NLP is "pre-train then fine-tune."

  Next: In Exercise 5, you'll build generative models (DCGAN + WGAN-GP)
  that CREATE new data instead of classifying existing data.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Complete Exercise 4")
print("=" * 70)
print(
)

